# ML Tutor — Week 7: Ensembles & friends
**Track:** Core ML · **Lab:** Lab 7 — Compare ensemble and margin-based models

**Objectives this week:**
1. Build ensemble models: random forest and gradient boosting (XGBoost). *(CO5)*
2. Apply KNN and SVM and compare model families on equal footing. *(CO2, CO5)*
3. Predict antibiotic resistance with a tuned, honestly evaluated ensemble. *(CO5)*

> Anti-shortcut rule: hints live in ML Tutor, revealed one tier at a time. Work here, check hints there. You are graded on unaided competence.


## Before you start (~25 min)
**Goal for this session:** Arrive ready to compare model families instead of worshipping one algorithm. We are building a stronger instinct for when a model family helps, when it overfits, and when feature scaling suddenly matters again. Reference delta: keep one tree ensemble as core; KNN/SVM comparison is optional stretch.

**Warm up — answer these from memory before scrolling on** (retrieval beats re-reading):
1. From Week 6, why do we keep tuning and evaluation separate from the test set?
2. What is one reason a model can look good on one metric and still be a poor practical choice?
3. Why is it useful to compare a family of models rather than only one favorite model?


In [ ]:
# SETUP — run me first (Shift+Enter)
import numpy as np
import pandas as pd

# Dataset: Synthetic hospital antibiogram table with fictional predictors such as age_band, prior_antibiotic_exposure, ward_type, prior_infection_count, comorbidity_index, recent_stay_length, lab_result_score, and resistance_flag. Course-generated teaching data only; no PHI.
# PLACEHOLDER: the dataset for this week arrives with the course repo under data/week-07/ .
# If a lab step expects a file, download it from the ml-tutor-labs repo's data/week-07/ folder first.

print("Setup OK — numpy", np.__version__, "· pandas", pd.__version__)


## Worked example — Random forests and bagging
**Lane A · the general idea:** A random forest grows many trees on bootstrapped samples and averages their results. That usually lowers variance and makes the model less brittle than a single tree.

**Lane B · the same idea in pharmacy terms:** For a resistance prediction task, a forest can capture nonlinear interactions among prior exposure, diagnosis history, and ward-level patterns without forcing one simple split to do all the work.

Below is a tiny, complete demo of this idea. Run it, read every line, change one thing, run it again.


In [ ]:
# WORKED EXAMPLE — a random forest is many trees VOTING (antibiotic resistance, toy)
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

df = pd.DataFrame({   # 6 synthetic admissions
    "prior_abx_courses": [0, 5, 1, 6, 0, 4],
    "icu_days":          [0, 12, 1, 8, 0, 10],
    "resistant":         [0, 1, 0, 1, 0, 1],
})
X, y = df[["prior_abx_courses", "icu_days"]], df["resistant"]
forest = RandomForestClassifier(n_estimators=25, random_state=0).fit(X, y)

new_pt = pd.DataFrame({"prior_abx_courses": [5], "icu_days": [9]})
print("P(resistant) =", forest.predict_proba(new_pt)[0][1])   # a VOTE SHARE across 25 trees


## 1. Build a baseline pipeline and evaluation split
Create one honest train/test split and fit a baseline classifier that can serve as the comparison anchor for all later models.

**Checkpoint:** Checkpoint 1 verifies: the split is stratified, preprocessing is inside the pipeline, and the baseline is fit without using test data during training.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# resistance_df: synthetic hospital antibiogram table (course-generated, no PHI).
resistance_df = pd.DataFrame({
    "age_band": ["18-40", "41-65", "65+", "41-65", "18-40", "65+", "41-65", "18-40",
                 "65+", "41-65", "18-40", "65+", "41-65", "18-40", "65+", "41-65"],
    "prior_antibiotic_exposure": [0, 2, 4, 1, 0, 3, 2, 1, 5, 2, 0, 4, 1, 0, 3, 2],
    "ward_type": ["general", "icu", "icu", "general", "general", "icu", "general", "general",
                  "icu", "general", "general", "icu", "general", "general", "icu", "general"],
    "prior_infection_count": [0, 1, 3, 0, 0, 2, 1, 0, 4, 1, 0, 2, 0, 0, 3, 1],
    "comorbidity_index": [1, 3, 5, 2, 1, 4, 3, 1, 5, 2, 1, 4, 2, 1, 4, 3],
    "recent_stay_length": [2, 5, 9, 3, 1, 7, 4, 2, 10, 3, 2, 8, 3, 1, 7, 4],
    "lab_result_score": [0.2, 0.6, 0.9, 0.3, 0.1, 0.8, 0.5, 0.2, 0.95, 0.4,
                          0.15, 0.85, 0.35, 0.1, 0.75, 0.45],
    "resistance_flag": [0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
})

target = "resistance_flag"
feature_cols = [
    "age_band",
    "prior_antibiotic_exposure",
    "ward_type",
    "prior_infection_count",
    "comorbidity_index",
    "recent_stay_length",
    "lab_result_score",
]

X = resistance_df[feature_cols]
y = resistance_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=7,
)

categorical_cols = ["age_band", "ward_type"]
numeric_cols = ["prior_antibiotic_exposure", "prior_infection_count", "comorbidity_index", "recent_stay_length", "lab_result_score"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_cols),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_cols),
    ]
)

baseline = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000)),
])

# TODO: fit the baseline on the training set only
...


**Self-check 1:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 2. Compare random forest and boosting
Train a random forest and a boosted tree model on the same data split, then compare their held-out performance to the baseline.

**Checkpoint:** Checkpoint 2 verifies: both tree-based models are fitted on the training set, test predictions are generated, and the student compares at least one metric against the baseline.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import recall_score, precision_score, accuracy_score

# resistance_df: synthetic hospital antibiogram table (course-generated, no PHI).
resistance_df = pd.DataFrame({
    "age_band": ["18-40", "41-65", "65+", "41-65", "18-40", "65+", "41-65", "18-40",
                 "65+", "41-65", "18-40", "65+", "41-65", "18-40", "65+", "41-65"],
    "prior_antibiotic_exposure": [0, 2, 4, 1, 0, 3, 2, 1, 5, 2, 0, 4, 1, 0, 3, 2],
    "ward_type": ["general", "icu", "icu", "general", "general", "icu", "general", "general",
                  "icu", "general", "general", "icu", "general", "general", "icu", "general"],
    "prior_infection_count": [0, 1, 3, 0, 0, 2, 1, 0, 4, 1, 0, 2, 0, 0, 3, 1],
    "comorbidity_index": [1, 3, 5, 2, 1, 4, 3, 1, 5, 2, 1, 4, 2, 1, 4, 3],
    "recent_stay_length": [2, 5, 9, 3, 1, 7, 4, 2, 10, 3, 2, 8, 3, 1, 7, 4],
    "lab_result_score": [0.2, 0.6, 0.9, 0.3, 0.1, 0.8, 0.5, 0.2, 0.95, 0.4,
                          0.15, 0.85, 0.35, 0.1, 0.75, 0.45],
    "resistance_flag": [0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
})

target = "resistance_flag"
feature_cols = ["age_band", "prior_antibiotic_exposure", "ward_type", "prior_infection_count",
                "comorbidity_index", "recent_stay_length", "lab_result_score"]
X = resistance_df[feature_cols]
y = resistance_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=7,
)

categorical_cols = ["age_band", "ward_type"]
numeric_cols = ["prior_antibiotic_exposure", "prior_infection_count", "comorbidity_index",
                "recent_stay_length", "lab_result_score"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_cols),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_cols),
    ]
)

baseline = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000)),
])
baseline.fit(X_train, y_train)

rf_model = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=...,
        random_state=7,
    )),
])

boost_model = Pipeline([
    ("prep", preprocess),
    ("clf", GradientBoostingClassifier(
        random_state=7,
    )),
])

# TODO: fit both models and compare their test-set metrics
...


**Self-check 2:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 3. Add KNN and SVM with the right preprocessing
Fit a distance-based model and a margin-based model so you can see why scaling is not optional for every family.

**Checkpoint:** Checkpoint 3 verifies: students fit KNN and SVM after preprocessing, generate predictions, and can explain why the scaler matters for these families.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

# resistance_df: synthetic hospital antibiogram table (course-generated, no PHI).
resistance_df = pd.DataFrame({
    "age_band": ["18-40", "41-65", "65+", "41-65", "18-40", "65+", "41-65", "18-40",
                 "65+", "41-65", "18-40", "65+", "41-65", "18-40", "65+", "41-65"],
    "prior_antibiotic_exposure": [0, 2, 4, 1, 0, 3, 2, 1, 5, 2, 0, 4, 1, 0, 3, 2],
    "ward_type": ["general", "icu", "icu", "general", "general", "icu", "general", "general",
                  "icu", "general", "general", "icu", "general", "general", "icu", "general"],
    "prior_infection_count": [0, 1, 3, 0, 0, 2, 1, 0, 4, 1, 0, 2, 0, 0, 3, 1],
    "comorbidity_index": [1, 3, 5, 2, 1, 4, 3, 1, 5, 2, 1, 4, 2, 1, 4, 3],
    "recent_stay_length": [2, 5, 9, 3, 1, 7, 4, 2, 10, 3, 2, 8, 3, 1, 7, 4],
    "lab_result_score": [0.2, 0.6, 0.9, 0.3, 0.1, 0.8, 0.5, 0.2, 0.95, 0.4,
                          0.15, 0.85, 0.35, 0.1, 0.75, 0.45],
    "resistance_flag": [0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0],
})

target = "resistance_flag"
feature_cols = ["age_band", "prior_antibiotic_exposure", "ward_type", "prior_infection_count",
                "comorbidity_index", "recent_stay_length", "lab_result_score"]
X = resistance_df[feature_cols]
y = resistance_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=7,
)

categorical_cols = ["age_band", "ward_type"]
numeric_cols = ["prior_antibiotic_exposure", "prior_infection_count", "comorbidity_index",
                "recent_stay_length", "lab_result_score"]

preprocess = ColumnTransformer(
    transformers=[
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore")),
        ]), categorical_cols),
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_cols),
    ]
)

knn_model = Pipeline([
    ("prep", preprocess),
    ("clf", KNeighborsClassifier(n_neighbors=...)),
])

svm_model = Pipeline([
    ("prep", preprocess),
    ("clf", SVC(kernel="rbf", probability=True, random_state=7)),
])

# TODO: fit, predict, and compare
...


**Self-check 3:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## 4. Choose a model and justify it
Rank the model families, name the winner with evidence, and write one plain-language justification for the antibiotic-resistance teaching task.

**Checkpoint:** Checkpoint 4 verifies: the student compares all contenders using the same evaluation logic and gives a non-technical justification for the selected family.


In [ ]:
# TODO: write a short comparison summary
# Include the baseline, random forest, boosting, KNN, and SVM.
# Name the one you would keep for the teaching case and explain why.

comparison_notes = """
...
"""

print(comparison_notes)


**Self-check 4:** run your cell, then compare its output against the checkpoint description above — does it satisfy *every* clause? Stuck? Graduated hints for this exact step live in **ML Tutor** (revealed one tier at a time, never the full answer). Fix, re-run, then move on.


## Reflection — make it stick (5 min, write before you leave)
1. **Teach it:** explain your Step 4 code to a colleague in exactly 3 sentences — what goes in, what happens, what comes out. Write the sentences below.
2. **What would break this?** A classmate insists: *“The fanciest model must be the best model.”* Using what you just built, describe one concrete case from this week's lab where acting on that belief produces a wrong result — and how you would catch it.


In [ ]:
# Your reflection (write as comments)
# 1) Three sentences:
#
# 2) What would break this, concretely:
#



## Optional challenge — Homework: Model-family comparison memo
Write a one-page memo comparing the baseline, random forest, boosting, KNN, and SVM on the antibiotic-resistance teaching task. Explain which family you would keep, which one you would reject, and why the result matters.

**Deliverable:** A short comparison memo plus a screenshot or note of the chosen metrics and model-ranking rationale.


In [ ]:
# HW / challenge workspace — build it here

